EN ESTE NOTEBOOK:
- Hacemos clustering con K-prototypes (agrupa conjuntos de datos que contienen una mezcla de variables numéricas y categóricas nominales). Para ello, tratamos 'ant_obstetrico' y 'ant_medico' como nominales.
- Hacemos clustering con K-Medoids.
- Hacemos clustering con HDBSCAN con distancia Gower (agrupa puntos de datos basados en la densidad de su distribución).
- Hacemos la descripción de los grupos en función de las variables Ecocardiografía | TA, carótida y oftálmica | Analítica | Scores categóricos | Antecedentes

In [23]:
import pandas as pd
import numpy as np

# Obtención de los datos

## Datos para clustering

In [24]:
df = pd.read_csv('imputado.csv')

In [25]:
df.columns

Index(['Unnamed: 0', 'id', 'peso_ini_gest', 'peso_fin_gest',
       'aumento_peso_gest', 'talla', 'imc_ini_gest', 'peso_rn', 'apgar_1min',
       'apgar_5min', 'apgar_10min', 'eg_parto', 'valor_sflt1',
       'eg_deter_sflt1_plgf', 'tas_1tri', 'tad_1tri', 'edad_materna_gest',
       'eg_eco_1tri', 'valor_plgf', 'ratio_sflt1_plgf', 'nivel_estudios',
       'riesgo_pe_1tri', 'ant_medico', 'ant_obstetrico', 'covid',
       'edema_agudo_pulmon', 'hemorragia_pospart_transfusion',
       'ini_trabajo_parto_espontaneo', 'parto_previo_mayor37_pre',
       'estudio_inicial_Angiocor', 'estudio_inicial_BiSC',
       'estudio_inicial_EUROPE', 'estudio_inicial_Ninguno',
       'etnia_Asia_Oriental', 'etnia_Blanca', 'etnia_Latina', 'etnia_Mixto',
       'etnia_Negra', 'etnia_Sureste_asiatico', 'concepcion_Espontanea',
       'concepcion_FIV', 'concepcion_FIV_ovodonacion',
       'concepcion_Inseminacion', 'tipo_parto_Cesarea', 'tipo_parto_Eutocico',
       'tipo_parto_Instrumentado', 'sexo_rn_Mascul

## Datos para descripción de los grupos

In [26]:
# Gestación y biomarcadores (hay que crear las variables MoM)
# El df que se usará para el clustering es df_modelo

def calcular_mom_variables(df):
    df = df.copy()
    # Definimos las condiciones basadas en 'eg_deter_sflt1_plgf'
    condiciones = [
        (df['eg_deter_sflt1_plgf'] >= 10) & (df['eg_deter_sflt1_plgf'] < 15), # 10 a 14.99
        (df['eg_deter_sflt1_plgf'] >= 15) & (df['eg_deter_sflt1_plgf'] < 20), # 15 a 19.99
        (df['eg_deter_sflt1_plgf'] >= 20) & (df['eg_deter_sflt1_plgf'] < 24),
        (df['eg_deter_sflt1_plgf'] >= 24) & (df['eg_deter_sflt1_plgf'] < 29),
        (df['eg_deter_sflt1_plgf'] >= 29) & (df['eg_deter_sflt1_plgf'] < 34),
        (df['eg_deter_sflt1_plgf'] >= 34) & (df['eg_deter_sflt1_plgf'] < 37),
        (df['eg_deter_sflt1_plgf'] >= 37)
]

    # Divisores para cada columna
    div_ratio = [24.8, 10.5, 4.92, 3.06, 3.75, 9.03, 19.6]
    div_sflt1 = [1328, 1355, 1299, 1355, 1742, 2552, 3485]
    div_plgf  = [52.6, 135, 264, 465, 471, 284, 191]

    # Aplicamos np.select para obtener el divisor correspondiente a cada fila
    # Si no cumple ninguna condición, devuelve NaN (np.nan)
    df['ratio_MoM'] = df['ratio_sflt1_plgf'] / np.select(condiciones, div_ratio, default=np.nan)
    df['sflt1_MoM'] = df['valor_sflt1'] / np.select(condiciones, div_sflt1, default=np.nan)
    df['plgf_MoM']  = df['valor_plgf'] / np.select(condiciones, div_plgf, default=np.nan)

    return df

df_modelo = calcular_mom_variables(df)

variables_a_borrar = ['ratio_sflt1_plgf', 'valor_sflt1', 'valor_plgf']
df_modelo = df_modelo.drop(columns=variables_a_borrar, errors='ignore')

In [27]:
# Ecocardiografía | TA, carótida y oftálmica | Analítica (df_comparacion)
df_clust2 = pd.read_csv('datos.csv')

In [28]:
# Scores categóricos (df_cat)

df_cat = pd.read_excel('variables_CARDIOMOM.xlsx')

scl_mapping = {
    'Nada': 0, 'Muy poco': 1, 'Poco': 2, 'Bastante': 3, 'Mucho': 4
}

# Funciones de categorización

def cat_regicor_dieta(score):
    if pd.isna(score): return np.nan
    if score < 23: return "Baja adherencia"
    elif 23 <= score <= 29: return "Adherencia moderada"
    else: return "Alta adherencia"

def cat_regicor_actividad(met):
    if pd.isna(met): return np.nan
    if met < 500: return "Actividad física baja"
    elif 500 <= met <= 1000: return "Actividad física moderada"
    else: return "Actividad física alta"

def cat_pss14(score):
    if pd.isna(score): return np.nan
    if score <= 13: return "Nivel de estrés bajo"
    elif 14 <= score <= 26: return "Nivel de estrés moderado"
    elif 27 <= score <= 40: return "Nivel de estrés alto"
    else: return "Nivel de estrés muy alto"

def cat_mfe30(score):
    if pd.isna(score): return np.nan
    if score < 8: return "Rendimiento de memoria óptimo"
    elif 9 <= score <= 35: return "Función normal (lapsus leves)"
    elif 36 <= score <= 50: return "Deterioro leve"
    else: return "Disfunción moderada-grave"

def cat_scl90r(t_score):
    if pd.isna(t_score): return np.nan
    if t_score >= 80: return "Patología severa"
    elif t_score >= 65: return "EN RIESGO"
    else: return "Normal / Sin riesgo"


df_cat['Cat_Dieta'] = df_cat['Score total'].apply(cat_regicor_dieta)
df_cat['Cat_Actividad'] = df_cat['Act física TOTAL (MET/semana)'].apply(cat_regicor_actividad)
df_cat['Cat_Estres'] = df_cat['Score'].apply(cat_pss14)
df_cat['Cat_Memoria'] = df_cat['Score.1'].apply(cat_mfe30)

# Procesamiento SCL-90-R
scl_items_cols = df_cat.loc[:, "Dolores de cabeza":"Pensar que en mi cabeza hay algo que no funciona bien"].columns
df_scl_numeric = df_cat[scl_items_cols].replace(scl_mapping).apply(pd.to_numeric, errors='coerce')
df_cat['SCL90_IGS_Raw'] = df_scl_numeric.mean(axis=1)

def calcular_t_score(row):
    raw = row['SCL90_IGS_Raw']
    if pd.isna(raw): return np.nan
    # Baremo Mujer (Media 0.17, DT 0.11) basado en el PDF
    return 50 + 10 * ((raw - 0.17) / 0.11)

df_cat['SCL90_T_Score'] = df_cat.apply(calcular_t_score, axis=1)
df_cat['Cat_SCL90R'] = df_cat['SCL90_T_Score'].apply(cat_scl90r)


cols_to_show = ['ID', 'Cat_Dieta', 'Cat_Actividad', 'Cat_Estres', 'Cat_Memoria', 'Cat_SCL90R']

print(df_cat[cols_to_show].head(20).to_string(index=False))

 ID           Cat_Dieta             Cat_Actividad               Cat_Estres                   Cat_Memoria          Cat_SCL90R
  1 Adherencia moderada     Actividad física alta     Nivel de estrés alto Función normal (lapsus leves)    Patología severa
  2 Adherencia moderada     Actividad física alta Nivel de estrés moderado Rendimiento de memoria óptimo    Patología severa
  3     Baja adherencia     Actividad física alta Nivel de estrés moderado                           NaN                 NaN
  4 Adherencia moderada                       NaN Nivel de estrés moderado                Deterioro leve    Patología severa
  5 Adherencia moderada                       NaN     Nivel de estrés alto Función normal (lapsus leves)    Patología severa
  6     Baja adherencia     Actividad física alta     Nivel de estrés alto                           NaN                 NaN
  7     Baja adherencia     Actividad física alta Nivel de estrés moderado                           NaN                 NaN


/var/folders/h2/lk1h3scx0p3cs7sk_k_sxylh0000gn/T/ipykernel_12371/459520726.py:51: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_scl_numeric = df_cat[scl_items_cols].replace(scl_mapping).apply(pd.to_numeric, errors='coerce')


In [29]:
# Antecedentes (df_clust2n)
# Nota: las variables 'ant_medico' y 'ant_obstetrico' ya están como nominales

cols_ant_obstetrico = [
    'parto_previo_menor37_pre', 'aborto_menor20', 
    'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp'
]
# Calculamos la suma y convertimos a 1 si es >= 1
df_clust2['ant_obstetrico'] = (df_clust2[cols_ant_obstetrico].sum(axis=1) >= 1).astype(int)

cols_ant_medico = [
    'ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 
    'enf_autoinm', 'fuma', 'alcohol', 'drogas'
]
# Calculamos la suma y convertimos a 1 si es >= 1
df_clust2['ant_medico'] = (df_clust2[cols_ant_medico].sum(axis=1) >= 1).astype(int)

# Trastorno hipertensivo: 1 si pe, sd_hellp o 'hipertension_gest' vs 0 si no tiene ninguna. 
cols_hipertension = ['pe', 'sd_hellp', 'hipertension_gest']
df_clust2['trastorno_hipertensivo'] = df_clust2[cols_hipertension].max(axis=1)

# Complicación placentaria: 'cir', 'peg', 'desprendimiento_placenta'
cols_placentaria = ['cir', 'peg', 'desprendimiento_placenta']
df_clust2['comp_placentaria'] = df_clust2[cols_placentaria].max(axis=1)

# Complicación materna grave: 'uci_materna_ucoi', 'hemocerebral_ictus', 'trombosis_venosa_prof'
cols_materna_grave = ['uci_materna_ucoi', 'hemocerebral_ictus', 'trombosis_venosa_prof']
df_clust2['comp_maternaGrave'] = df_clust2[cols_materna_grave].max(axis=1)

# Complicación metabólica: 'diabetes_gest', 'colestasis_intrahepatica'
cols_metabolica = ['diabetes_gest', 'colestasis_intrahepatica']
df_clust2['comp_metabolica'] = df_clust2[cols_metabolica].max(axis=1)

In [30]:
df_clust2['ant_medico'].unique()

array([0, 1])

In [31]:
df_clust2['ant_obstetrico'].unique()

array([0, 1])

# Clustering con K-prototypes (numéricas + categóricas)

Las variables consideradas en el clustering son ['peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM', 'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 
'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica']

In [32]:
pip install kmodes


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
pip install gower


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [34]:
from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import gower
import matplotlib.pyplot as plt

In [35]:
# Definición de variables
variables_numericas = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest',
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto',
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM'
]

variables_categoricas = [
    'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo',
    'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica'
]

todas_variables = variables_numericas + variables_categoricas

# Candidatas (todas menos 'id')
candidatas_num = list(variables_numericas)
candidatas_cat = list(variables_categoricas)


# Función de evaluación MODIFICADA para devolver n
def evaluar_kproto(vars_num, vars_cat, k, df_modelo):
    todas = vars_num + vars_cat

    X_num_loc = StandardScaler().fit_transform(df_modelo[vars_num]) if vars_num else np.empty((len(df_modelo), 0))
    X_cat_loc = df_modelo[vars_cat].values.astype(str) if vars_cat else np.empty((len(df_modelo), 0))
    X_loc = np.concatenate([X_num_loc, X_cat_loc], axis=1)
    cat_cols_loc = list(range(len(vars_num), len(todas)))

    X_gow = df_modelo[todas].copy()
    if vars_cat:
        X_gow[vars_cat] = X_gow[vars_cat].astype(int)
    dist_loc = gower.gower_matrix(X_gow)

    kp = KPrototypes(n_clusters=k, init='Cao', n_init=3, random_state=42)
    labels = kp.fit_predict(X_loc, categorical=cat_cols_loc)
    
    # Calcular tamaños de los grupos
    counts = pd.Series(labels).value_counts().sort_index().tolist()
    
    sil = silhouette_score(dist_loc, labels, metric='precomputed')
    return sil, counts


# Forward selection para K-prototypes
def find_top3_kproto(df_modelo, k_range):
    all_k_results = []

    for k in k_range:
        print(f"\nBuscando mejores combinaciones para k={k}...")
        combinations_tested = []
        current_num = []
        current_cat = []
        remaining_num = list(candidatas_num)
        remaining_cat = list(candidatas_cat)

        while remaining_num or remaining_cat:
            step_scores = []

            # Probar añadir cada variable candidata (numérica o categórica)
            candidatas_paso = (
                [('num', v) for v in remaining_num] +
                [('cat', v) for v in remaining_cat]
            )

            for tipo, var in candidatas_paso:
                vars_num_tmp = current_num + ([var] if tipo == 'num' else [])
                vars_cat_tmp = current_cat + ([var] if tipo == 'cat' else [])

                # Necesitamos al menos 2 variables para calcular Silhouette
                if len(vars_num_tmp) + len(vars_cat_tmp) < 2:
                    step_scores.append((None, var, tipo))
                    continue

                try:
                    sil, counts = evaluar_kproto(vars_num_tmp, vars_cat_tmp, k, df_modelo)
                    combinations_tested.append({
                        'k': k,
                        'silhouette': sil,
                        'num_variables': len(vars_num_tmp) + len(vars_cat_tmp),
                        'variables': vars_num_tmp + vars_cat_tmp,
                        'n_grupos': counts # Guardamos los tamaños
                    })
                    step_scores.append((sil, var, tipo))
                except Exception:
                    step_scores.append((None, var, tipo))

            # Elegir la mejor variable del paso
            step_scores_validos = [(s, v, t) for s, v, t in step_scores if s is not None]

            if not step_scores_validos:
                tipo, var = candidatas_paso[0][0], candidatas_paso[0][1]
                if tipo == 'num':
                    current_num.append(var)
                    remaining_num.remove(var)
                else:
                    current_cat.append(var)
                    remaining_cat.remove(var)
                continue

            step_scores_validos.sort(key=lambda x: x[0], reverse=True)
            best_sil, best_var, best_tipo = step_scores_validos[0]

            if best_tipo == 'num':
                current_num.append(best_var)
                remaining_num.remove(best_var)
            else:
                current_cat.append(best_var)
                remaining_cat.remove(best_var)

            print(f"  Paso {len(current_num)+len(current_cat)}: +{best_var:<30} → Sil = {best_sil:.4f}")

        # Top 3 para este k
        if combinations_tested:
            df_k = pd.DataFrame(combinations_tested)
            top3 = df_k.sort_values('silhouette', ascending=False).head(3)
            all_k_results.append(top3)
            print(f"\n  Top 3 para k={k}:")
            for _, r in top3.iterrows():
                print(f"    Sil={r['silhouette']:.4f} | Grupos: {r['n_grupos']} | {r['num_variables']} vars: {r['variables']}")

    return pd.concat(all_k_results, ignore_index=True)


# Ejecutar
top3_results_mixto = find_top3_kproto(df_modelo, range(2, 6)) # Rango reducido para ejemplo

print("\n" + "="*70)
print("RESUMEN: Top 3 combinaciones por k")
print("="*70)
pd.set_option('display.max_colwidth', None)
display(top3_results_mixto[['k', 'silhouette', 'n_grupos', 'variables']])


Buscando mejores combinaciones para k=2...
  Paso 2: +ant_medico                     → Sil = 0.4497
  Paso 3: +apgar_10min                    → Sil = 0.6822
  Paso 4: +apgar_5min                     → Sil = 0.7149
  Paso 5: +apgar_1min                     → Sil = 0.7222
  Paso 6: +eg_parto                       → Sil = 0.7075
  Paso 7: +comp_maternaGrave              → Sil = 0.7057
  Paso 8: +peso_rn                        → Sil = 0.6811
  Paso 9: +comp_placentaria               → Sil = 0.6544
  Paso 10: +tad_1tri                       → Sil = 0.6324
  Paso 11: +aumento_peso_gest              → Sil = 0.6090
  Paso 12: +imc_ini_gest                   → Sil = 0.5812
  Paso 13: +tas_1tri                       → Sil = 0.5607
  Paso 14: +talla                          → Sil = 0.5367
  Paso 15: +peso_fin_gest                  → Sil = 0.5189
  Paso 16: +comp_metabolica                → Sil = 0.4948
  Paso 17: +trastorno_hipertensivo         → Sil = 0.4728
  Paso 18: +ant_obstetrico          

,k,silhouette,n_grupos,variables
0,2,0.722224,"[37, 422]","[peso_ini_gest, apgar_10min, apgar_5min, apgar_1min, ant_medico]"
1,2,0.714869,"[35, 424]","[peso_ini_gest, apgar_10min, apgar_5min, ant_medico]"
2,2,0.707450,"[31, 428]","[peso_ini_gest, apgar_10min, apgar_5min, apgar_1min, eg_parto, ant_medico]"
3,3,0.498643,"[103, 33, 323]","[peso_ini_gest, apgar_10min, imc_ini_gest, comp_maternaGrave]"
4,3,0.488487,"[103, 322, 34]","[peso_ini_gest, apgar_10min, imc_ini_gest, apgar_5min, comp_maternaGrave]"
5,3,0.486609,"[111, 315, 33]","[peso_ini_gest, apgar_10min, imc_ini_gest, peso_fin_gest, comp_maternaGrave]"
6,4,0.466029,"[10, 27, 313, 109]","[peso_ini_gest, peso_fin_gest, imc_ini_gest, apgar_10min, apgar_5min, apgar_1min, ant_medico]"
7,4,0.417299,"[10, 27, 313, 109]","[peso_ini_gest, peso_fin_gest, imc_ini_gest, apgar_10min, apgar_5min, apgar_1min, ant_medico, comp_metabolica]"
8,4,0.415081,"[10, 27, 313, 109]","[peso_ini_gest, peso_fin_gest, imc_ini_gest, apgar_10min, apgar_5min, apgar_1min, ant_medico, comp_maternaGrave]"
9,5,0.399129,"[92, 59, 152, 135, 21]","[peso_ini_gest, ant_medico]"


# Descripción de los grupos

In [36]:
from scipy import stats
from itertools import combinations

In [37]:
vars_embarazo = [c for c in [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest',
    'edad_materna_gest', 'tas_1tri', 'tad_1tri', 'eg_eco_1tri', 'eg_parto',
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min'
] if c in df_modelo.columns]

vars_biomarcadores = [c for c in ['ratio_MoM', 'sflt1_MoM', 'plgf_MoM'] if c in df_modelo.columns]

vars_eco = [c for c in [
    'diam_telediastolico', 'diam_telesistolico', 'dtsvi_indexado', 'septo_iv_diastole',
    'pared_posterior_vi_diastole', 'diam_ai', 'ai_volumen', 'ad_volumen', 'tapse',
    'e_mitral', 'e_e', 'gc_tsvi', 'urato_acidourico'
] if c in df_clust2.columns]

vars_ta_car_oft = [c for c in [
    'ta_sistolica', 'ta_diastolica', 'frec_cardiaca',
    'right_1peak_systolic_velocity', 'right_2peak_systolic_velocity',
    'right_pulsatility_index', 'right_psv_ratio',
    'left_1peak_systolic_velocity', 'left_2peak_systolic_velocity',
    'left_pulsatility_index', 'left_psv_ratio', 'right_mean', 'left_mean'
] if c in df_clust2.columns]

vars_analitica = [c for c in [
    'plgf_pg', 'troponina_t', 'nt_probnp', 'hemoglobina', 'hematocrito',
    'leucocitos', 'plaquetas', 'glucosa', 'sodio', 'potasio',
    'urato_acidourico', 'creatinina', 'hemoglobina_glicada', 'ast', 'alt',
    'bilirubina_total', 'ldh', 'vldl', 'ldl', 'hdl', 'colesterol_total',
    'trigliceridos', 'prote_totales_orina', 'albumina_orina',
    'ratio_albumina_creatinina', 'tirotropina', 'prolactina'
] if c in df_clust2.columns]

vars_antecedentes = [c for c in [
    'trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave',
    'comp_metabolica', 'ant_obstetrico', 'ant_medico'
] if c in df_clust2.columns]

cols_cat_scores = ['ID', 'Cat_Dieta', 'Cat_Actividad', 'Cat_Estres', 'Cat_Memoria', 'Cat_SCL90R']

grupos_informe = {
    "GESTACIÓN Y BIOMARCADORES": vars_embarazo + vars_biomarcadores,
    "ECOCARDIOGRAFÍA":           vars_eco,
    "TA, CARÓTIDA Y OFTÁLMICA":  vars_ta_car_oft,
    "ANALÍTICA":                 vars_analitica,
    "SCORES CATEGÓRICOS":        [c for c in cols_cat_scores if c != 'ID' and c in df_cat.columns],
    "ANTECEDENTES":              vars_antecedentes,
}

# Construcción del dataframe maestro

# Asegurar tipos consistentes
df_modelo['id'] = df_modelo['id'].astype(float)
df_clust2['id'] = df_clust2['id'].astype(float)
df_cat['ID']    = df_cat['ID'].astype(float)

# Eliminación de duplicados
df_clust_dedup  = df_modelo.drop_duplicates(subset='id', keep='first')
df_clust2_dedup = df_clust2.drop_duplicates(subset='id', keep='first')
df_cat_dedup    = df_cat.drop_duplicates(subset='ID', keep='first')

# Iniciamos df_maestro
df_maestro = df_clust_dedup.copy()

# MERGE 1: Unir variables de df_clust2 (EVITANDO DUPLICADOS QUE CAUSAN MERGEERROR)
cols_interes_clust2 = vars_eco + vars_ta_car_oft + vars_analitica + vars_antecedentes
# Solo traemos columnas que NO existan ya en df_maestro para evitar colisiones
cols_nuevas_clust2 = [c for c in cols_interes_clust2 if c in df_clust2_dedup.columns and c not in df_maestro.columns]

df_maestro = pd.merge(
    df_maestro, 
    df_clust2_dedup[['id'] + cols_nuevas_clust2], 
    on='id', 
    how='left'
)

# MERGE 2: Unir variables de scores categóricos
cols_cat_existentes = [c for c in cols_cat_scores if c in df_cat_dedup.columns]
cols_cat_nuevas = [c for c in cols_cat_existentes if c != 'ID' and c not in df_maestro.columns]

df_maestro = pd.merge(
    df_maestro,
    df_cat_dedup[['ID'] + cols_cat_nuevas],
    left_on='id', right_on='ID',
    how='left'
).drop(columns=['ID'], errors='ignore')

# Asegurar que los antecedentes sean tratados como categorías
vars_antecedentes = [v for v in vars_antecedentes if v in df_maestro.columns]
for v in vars_antecedentes:
    df_maestro[v] = df_maestro[v].astype(str).replace('nan', np.nan).astype('category')

grupos_informe["ANTECEDENTES"] = vars_antecedentes
assert len(df_maestro) == len(df_clust_dedup), "ERROR: df_maestro tiene duplicados inesperados"

# Funciones auxiliares

def describir_clusters(df, variables_num, variables_cat):
    partes = []
    if variables_num:
        # Mejora: Manejo de grupos con n=1 para evitar errores en std()
        res_num = df.groupby('cluster')[variables_num].agg(
            lambda x: f"{x.mean():.2f} ± {x.std():.2f}" if len(x.dropna()) > 1 else f"{x.mean():.2f} ± 0.00"
        ).T
        res_num.columns = [f"Grupo {i}" for i in res_num.columns]
        partes.append(res_num)

    if variables_cat:
        cat_list = []
        for col in variables_cat:
            valid = df[[col, 'cluster']].dropna(subset=[col]).copy()
            valid[col] = valid[col].astype(str)
            try:
                ct_n   = pd.crosstab(valid[col], valid['cluster'])
                ct_pct = pd.crosstab(valid[col], valid['cluster'], normalize='columns') * 100
                ct_fmt = ct_n.astype(str).copy()
                for r in ct_n.index:
                    for c in ct_n.columns:
                        ct_fmt.loc[r, c] = f"{int(ct_n.loc[r, c])} ({ct_pct.loc[r, c]:.1f}%)"
                ct_fmt.index = [f"{col}: {idx}" for idx in ct_fmt.index]
                cat_list.append(ct_fmt)
            except Exception as e:
                print(f"  [AVISO] No se pudo tabular '{col}': {e}")
        if cat_list:
            res_cat = pd.concat(cat_list)
            res_cat.columns = [f"Grupo {i}" for i in res_cat.columns]
            partes.append(res_cat)
    return pd.concat(partes) if partes else pd.DataFrame()

def realizar_analisis_avanzado(df, var, grupo_col):
    serie = df[var]
    if hasattr(serie, 'values') and serie.values.ndim > 1:
        df = df.copy()
        df[var] = serie.values[:, 0]

    grupos_ids = sorted(df[grupo_col].unique())
    dict_datos = {g: df[df[grupo_col] == g][var].dropna() for g in grupos_ids}
    n_grupos = len(grupos_ids)

    datos_validos = [d for d in dict_datos.values() if len(d) > 0]
    if len(datos_validos) < 2:
        return [var, "N/A", "N/A", "N/A"] + ["-"] * n_grupos

    es_numerica = pd.api.types.is_numeric_dtype(df[var]) and not set(df[var].dropna().unique()).issubset({0, 1, '0', '1', '0.0', '1.0'})

    if es_numerica:
        p_shapiro = stats.shapiro(df[var].dropna())[1] if len(df[var].dropna()) > 3 else 0
        skew = df[var].skew()
        kurt = df[var].kurtosis()
        es_normal = (p_shapiro > 0.05) and (abs(skew) < 1) and (abs(kurt) < 3)
        
        # Levene requiere al menos 2 datos por grupo
        datos_levene = [d for d in datos_validos if len(d) > 1]
        p_levene = stats.levene(*datos_levene)[1] if len(datos_levene) > 1 else 1
        homogeneas = p_levene > 0.05

        if n_grupos == 2:
            test_name = "Prueba T" if es_normal else "U Mann-Whitney"
            g1, g2 = list(dict_datos.values())[0], list(dict_datos.values())[1]
            if es_normal:
                _, p_val_total = stats.ttest_ind(g1, g2, equal_var=homogeneas)
            else:
                _, p_val_total = stats.mannwhitneyu(g1, g2)
        else:
            if es_normal:
                test_name = "ANOVA" if homogeneas else "Welch ANOVA"
                p_val_total = stats.f_oneway(*datos_validos)[1] if homogeneas else stats.alexandergovern(*datos_validos).pvalue
            else:
                test_name = "Kruskal-Wallis"
                _, p_val_total = stats.kruskal(*datos_validos)
        metrica = "media ± DE" if es_normal else "mediana ± IQR"
    else:
        serie_clean = df[var].dropna().astype(str)
        contingencia = pd.crosstab(serie_clean, df.loc[serie_clean.index, grupo_col])
        test_name = "Chi-Square"
        _, p_val_total, _, _ = stats.chi2_contingency(contingencia)

    # Post-hoc pairwise
    pairs = list(combinations(grupos_ids, 2))
    p_pairs = []
    for g1_id, g2_id in pairs:
        d1, d2 = dict_datos[g1_id], dict_datos[g2_id]
        if len(d1) > 1 and len(d2) > 1:
            if es_numerica:
                _, p_p = stats.mannwhitneyu(d1, d2)
            else:
                sub_cont = pd.crosstab(df[df[grupo_col].isin([g1_id, g2_id])][var].astype(str), 
                                       df[df[grupo_col].isin([g1_id, g2_id])][grupo_col])
                _, p_p, _, _ = stats.chi2_contingency(sub_cont)
            p_adj = min(p_p * len(pairs), 1.0)
            p_pairs.append(f"G{g1_id}vG{g2_id}: {p_adj:.3f}")
        else:
            p_pairs.append(f"G{g1_id}vG{g2_id}: N/A")
    
    res_grupos = []
    for g in grupos_ids:
        g_data = dict_datos[g]
        if len(g_data) == 0: res_grupos.append("N/A")
        elif es_numerica:
            if metrica == "media ± DE":
                res_grupos.append(f"{g_data.mean():.2f} ± {g_data.std():.2f}")
            else:
                res_grupos.append(f"{g_data.median():.2f} ± {g_data.quantile(0.75)-g_data.quantile(0.25):.2f}")
        else:
            counts = g_data.astype(str).value_counts().sort_index()
            total_n = len(g_data)
            res_grupos.append(" | ".join([f"{c}: n={int(n)} ({n/total_n*100:.1f}%)" for c, n in counts.items()]))

    return [var, test_name, f"{p_val_total:.4f}", " | ".join(p_pairs)] + res_grupos

# Bucle principal de ejecución
vars_forzar_categorica = ['trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave', 
                          'comp_metabolica', 'ant_obstetrico', 'ant_medico']

resultados_filtrados = top3_results_mixto[top3_results_mixto['k'].isin([2, 3, 4, 5])]
pd.set_option('display.max_colwidth', None)

for idx, row in resultados_filtrados.iterrows():
    k_actual = int(row['k'])
    vars_clustering = row['variables']
    print(f"\n{'='*70}\nCOMBINACIÓN {idx+1} | K = {k_actual} | SILHOUETTE = {row['silhouette']:.4f}\n{'='*70}")

    # Ejecución K-Prototypes
    vars_num_cl = [v for v in vars_clustering if v in variables_numericas]
    vars_cat_cl = [v for v in vars_clustering if v in variables_categoricas]
    X_num_cl = StandardScaler().fit_transform(df_modelo[vars_num_cl]) if vars_num_cl else np.empty((len(df_modelo), 0))
    X_cat_cl = df_modelo[vars_cat_cl].values.astype(str) if vars_cat_cl else np.empty((len(df_modelo), 0))
    X_cl = np.concatenate([X_num_cl, X_cat_cl], axis=1)
    cat_indices = list(range(len(vars_num_cl), X_cl.shape[1]))

    kp = KPrototypes(n_clusters=k_actual, init='Cao', n_init=10, random_state=42)
    labels = kp.fit_predict(X_cl, categorical=cat_indices)

    # Preparar df_desc para esta iteración
    mapa = pd.DataFrame({'id': df_modelo['id'].values, 'cluster': labels})
    df_desc = df_maestro.drop(columns=['cluster'], errors='ignore').merge(mapa, on='id', how='inner')
    df_desc = df_desc.drop_duplicates(subset='id').loc[:, ~df_desc.columns.duplicated()]

    print(f"\nTAMAÑO DE GRUPOS:\n{df_desc['cluster'].value_counts().sort_index().to_frame().T}")

    # Bloques de descripción y estadística
    for nombre_bloque, vars_bloque in grupos_informe.items():
        v_num = [v for v in vars_bloque if v in df_desc.columns and v not in vars_forzar_categorica and pd.api.types.is_numeric_dtype(df_desc[v])]
        v_cat = [v for v in vars_bloque if v in df_desc.columns and (v in vars_forzar_categorica or not pd.api.types.is_numeric_dtype(df_desc[v]))]
        
        if v_num or v_cat:
            print(f"\n{nombre_bloque}")
            display(describir_clusters(df_desc, v_num, v_cat))
            
            data_stats = [realizar_analisis_avanzado(df_desc, v, 'cluster') for v in vars_bloque if v in df_desc.columns and v not in ['id', 'ID']]
            data_stats = [r for r in data_stats if r]
            if data_stats:
                headers = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"] + [f"G{i}" for i in range(k_actual)]
                display(pd.DataFrame(data_stats, columns=headers))


COMBINACIÓN 1 | K = 2 | SILHOUETTE = 0.7222

TAMAÑO DE GRUPOS:
cluster   0    1
count    37  422

GESTACIÓN Y BIOMARCADORES


,Grupo 0,Grupo 1
peso_ini_gest,72.14 ± 17.02,66.39 ± 14.22
peso_fin_gest,84.36 ± 17.97,78.08 ± 14.83
aumento_peso_gest,12.31 ± 6.22,11.67 ± 5.43
talla,162.04 ± 5.51,163.98 ± 6.22
imc_ini_gest,27.46 ± 6.59,24.67 ± 4.96
edad_materna_gest,36.93 ± 4.50,35.41 ± 4.67
tas_1tri,119.86 ± 13.40,113.87 ± 11.29
tad_1tri,76.63 ± 10.63,72.73 ± 8.02
eg_eco_1tri,12.79 ± 0.50,12.73 ± 0.61
eg_parto,35.84 ± 3.95,39.31 ± 1.71


,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1
0,peso_ini_gest,U Mann-Whitney,0.0369,G0vG1: 0.037,67.00 ± 19.00,63.70 ± 17.00
1,peso_fin_gest,U Mann-Whitney,0.0369,G0vG1: 0.037,83.00 ± 21.00,76.00 ± 17.00
2,aumento_peso_gest,U Mann-Whitney,0.7785,G0vG1: 0.778,12.00 ± 5.50,11.35 ± 5.30
3,talla,U Mann-Whitney,0.0679,G0vG1: 0.068,162.00 ± 9.50,164.00 ± 8.00
4,imc_ini_gest,U Mann-Whitney,0.0045,G0vG1: 0.005,24.97 ± 7.90,23.47 ± 6.37
5,edad_materna_gest,U Mann-Whitney,0.0678,G0vG1: 0.068,36.60 ± 5.89,35.48 ± 6.41
6,tas_1tri,U Mann-Whitney,0.0046,G0vG1: 0.005,118.00 ± 17.33,113.02 ± 14.00
7,tad_1tri,U Mann-Whitney,0.0029,G0vG1: 0.003,77.00 ± 7.35,72.41 ± 9.00
8,eg_eco_1tri,U Mann-Whitney,0.6157,G0vG1: 0.616,12.80 ± 0.44,12.79 ± 0.50
9,eg_parto,U Mann-Whitney,0.0000,G0vG1: 0.000,37.20 ± 5.30,39.60 ± 2.00



ECOCARDIOGRAFÍA


,Grupo 0,Grupo 1
diam_telediastolico,44.59 ± 4.90,44.99 ± 5.48
diam_telesistolico,28.80 ± 4.35,29.96 ± 4.53
dtsvi_indexado,15.65 ± 3.03,17.00 ± 2.72
septo_iv_diastole,11.47 ± 14.65,9.22 ± 8.98
pared_posterior_vi_diastole,9.52 ± 9.78,8.47 ± 8.38
diam_ai,34.85 ± 4.78,33.28 ± 4.61
ai_volumen,39.49 ± 12.69,36.88 ± 13.27
ad_volumen,33.12 ± 12.83,32.73 ± 15.52
tapse,26.49 ± 3.91,26.02 ± 17.50
e_mitral,75.18 ± 16.17,79.47 ± 62.85


...

,Grupo 0,Grupo 1,Grupo 2,Grupo 3,Grupo 4
Cat_Dieta: Adherencia moderada,40 (47.6%),43 (62.3%),11 (52.4%),98 (66.2%),93 (67.9%)
Cat_Dieta: Alta adherencia,2 (2.4%),5 (7.2%),0 (0.0%),12 (8.1%),6 (4.4%)
Cat_Dieta: Baja adherencia,42 (50.0%),21 (30.4%),10 (47.6%),38 (25.7%),38 (27.7%)
Cat_Actividad: Actividad física alta,71 (84.5%),63 (95.5%),19 (90.5%),137 (92.6%),123 (91.8%)
Cat_Actividad: Actividad física baja,1 (1.2%),0 (0.0%),1 (4.8%),3 (2.0%),6 (4.5%)
Cat_Actividad: Actividad física moderada,12 (14.3%),3 (4.5%),1 (4.8%),8 (5.4%),5 (3.7%)
Cat_Estres: Nivel de estrés alto,34 (41.0%),26 (37.7%),10 (47.6%),50 (33.8%),42 (30.9%)
Cat_Estres: Nivel de estrés bajo,10 (12.0%),8 (11.6%),3 (14.3%),16 (10.8%),15 (11.0%)
Cat_Estres: Nivel de estrés moderado,33 (39.8%),33 (47.8%),7 (33.3%),79 (53.4%),76 (55.9%)
Cat_Estres: Nivel de estrés muy alto,6 (7.2%),2 (2.9%),1 (4.8%),3 (2.0%),3 (2.2%)


,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1,G2,G3,G4
0,Cat_Dieta,Chi-Square,0.0048,G0vG1: 0.303 | G0vG2: 1.000 | G0vG3: 0.005 | G0vG4: 0.036 | G1vG2: 1.000 | G1vG3: 1.000 | G1vG4: 1.000 | G2vG3: 0.683 | G2vG4: 1.000 | G3vG4: 1.000,Adherencia moderada: n=40 (47.6%) | Alta adherencia: n=2 (2.4%) | Baja adherencia: n=42 (50.0%),Adherencia moderada: n=43 (62.3%) | Alta adherencia: n=5 (7.2%) | Baja adherencia: n=21 (30.4%),Adherencia moderada: n=11 (52.4%) | Baja adherencia: n=10 (47.6%),Adherencia moderada: n=98 (66.2%) | Alta adherencia: n=12 (8.1%) | Baja adherencia: n=38 (25.7%),Adherencia moderada: n=93 (67.9%) | Alta adherencia: n=6 (4.4%) | Baja adherencia: n=38 (27.7%)
1,Cat_Actividad,Chi-Square,0.0436,G0vG1: 0.369 | G0vG2: 1.000 | G0vG3: 0.637 | G0vG4: 0.101 | G1vG2: 1.000 | G1vG3: 0.472 | G1vG4: 1.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 1.000,Actividad física alta: n=71 (84.5%) | Actividad física baja: n=1 (1.2%) | Actividad física moderada: n=12 (14.3%),Actividad física alta: n=63 (95.5%) | Actividad física moderada: n=3 (4.5%),Actividad física alta: n=19 (90.5%) | Actividad física baja: n=1 (4.8%) | Actividad física moderada: n=1 (4.8%),Actividad física alta: n=137 (92.6%) | Actividad física baja: n=3 (2.0%) | Actividad física moderada: n=8 (5.4%),Actividad física alta: n=123 (91.8%) | Actividad física baja: n=6 (4.5%) | Actividad física moderada: n=5 (3.7%)
2,Cat_Estres,Chi-Square,0.4133,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 0.811 | G0vG4: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G1vG4: 1.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 1.000,Nivel de estrés alto: n=34 (41.0%) | Nivel de estrés bajo: n=10 (12.0%) | Nivel de estrés moderado: n=33 (39.8%) | Nivel de estrés muy alto: n=6 (7.2%),Nivel de estrés alto: n=26 (37.7%) | Nivel de estrés bajo: n=8 (11.6%) | Nivel de estrés moderado: n=33 (47.8%) | Nivel de estrés muy alto: n=2 (2.9%),Nivel de estrés alto: n=10 (47.6%) | Nivel de estrés bajo: n=3 (14.3%) | Nivel de estrés moderado: n=7 (33.3%) | Nivel de estrés muy alto: n=1 (4.8%),Nivel de estrés alto: n=50 (33.8%) | Nivel de estrés bajo: n=16 (10.8%) | Nivel de estrés moderado: n=79 (53.4%) | Nivel de estrés muy alto: n=3 (2.0%),Nivel de estrés alto: n=42 (30.9%) | Nivel de estrés bajo: n=15 (11.0%) | Nivel de estrés moderado: n=76 (55.9%) | Nivel de estrés muy alto: n=3 (2.2%)
3,Cat_Memoria,Chi-Square,0.5060,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G0vG4: 1.000 | G1vG2: 1.000 | G1vG3: 0.982 | G1vG4: 1.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 1.000,Deterioro leve: n=13 (15.7%) | Disfunción moderada-grave: n=7 (8.4%) | Función normal (lapsus leves): n=52 (62.7%) | Rendimiento de memoria óptimo: n=11 (13.3%),Deterioro leve: n=12 (17.9%) | Disfunción moderada-grave: n=4 (6.0%) | Función normal (lapsus leves): n=38 (56.7%) | Rendimiento de memoria óptimo: n=13 (19.4%),Deterioro leve: n=5 (23.8%) | Disfunción moderada-grave: n=4 (19.0%) | Función normal (lapsus leves): n=8 (38.1%) | Rendimiento de memoria óptimo: n=4 (19.0%),Deterioro leve: n=16 (10.8%) | Disfunción moderada-grave: n=18 (12.2%) | Función normal (lapsus leves): n=86 (58.1%) | Rendimiento de memoria óptimo: n=28 (18.9%),Deterioro leve: n=19 (14.1%) | Disfunción moderada-grave: n=11 (8.1%) | Función normal (lapsus leves): n=76 (56.3%) | Rendimiento de memoria óptimo: n=29 (21.5%)
4,Cat_SCL90R,Chi-Square,0.0902,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 0.324 | G0vG4: 1.000 | G1vG2: 1.000 | G1vG3: 0.248 | G1vG4: 1.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 1.000,EN RIESGO: n=10 (12.3%) | Normal / Sin riesgo: n=11 (13.6%) | Patología severa: n=60 (74.1%),EN RIESGO: n=12 (17.4%) | Normal / Sin riesgo: n=7 (10.1%) | Patología severa: n=50 (72.5%),EN RIESGO: n=2 (9.5%) | Normal / Sin riesgo: n=3 (14.3%) | Patología severa: n=16 (76.2%),EN RIESGO: n=17 (11.6%) | Normal / Sin riesgo: n=41 (27.9%) | Patología severa: n=89 (60.5%),EN RIESGO: n=16 (12.0%) | Normal / Sin riesgo: n=27 (20.3%) | Patología severa: n=90 (67.7%)



ANTECEDENTES


,Grupo 0,Grupo 1,Grupo 2,Grupo 3,Grupo 4
trastorno_hipertensivo: 0.0,56 (66.7%),0 (0.0%),12 (57.1%),148 (100.0%),111 (81.0%)
trastorno_hipertensivo: 1.0,28 (33.3%),69 (100.0%),9 (42.9%),0 (0.0%),26 (19.0%)
comp_placentaria: 0.0,76 (90.5%),55 (79.7%),18 (85.7%),132 (89.2%),109 (79.6%)
comp_placentaria: 1.0,8 (9.5%),14 (20.3%),3 (14.3%),16 (10.8%),28 (20.4%)
comp_maternaGrave: 0.0,77 (91.7%),45 (65.2%),20 (95.2%),146 (98.6%),130 (94.9%)
comp_maternaGrave: 1.0,7 (8.3%),24 (34.8%),1 (4.8%),2 (1.4%),7 (5.1%)
comp_metabolica: 0.0,69 (82.1%),63 (91.3%),18 (85.7%),138 (93.2%),123 (89.8%)
comp_metabolica: 1.0,15 (17.9%),6 (8.7%),3 (14.3%),10 (6.8%),14 (10.2%)
ant_obstetrico: -1,61 (72.6%),52 (75.4%),17 (81.0%),106 (71.6%),100 (73.0%)
ant_obstetrico: 1,18 (21.4%),15 (21.7%),2 (9.5%),34 (23.0%),30 (21.9%)


,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1,G2,G3,G4
0,trastorno_hipertensivo,Chi-Square,0.0000,G0vG1: 0.000 | G0vG2: 1.000 | G0vG3: 0.000 | G0vG4: 0.245 | G1vG2: 0.000 | G1vG3: 0.000 | G1vG4: 0.000 | G2vG3: 0.000 | G2vG4: 0.299 | G3vG4: 0.000,0.0: n=56 (66.7%) | 1.0: n=28 (33.3%),1.0: n=69 (100.0%),0.0: n=12 (57.1%) | 1.0: n=9 (42.9%),0.0: n=148 (100.0%),0.0: n=111 (81.0%) | 1.0: n=26 (19.0%)
1,comp_placentaria,Chi-Square,0.0691,G0vG1: 0.975 | G0vG2: 1.000 | G0vG3: 1.000 | G0vG4: 0.518 | G1vG2: 1.000 | G1vG3: 0.944 | G1vG4: 1.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 0.372,0.0: n=76 (90.5%) | 1.0: n=8 (9.5%),0.0: n=55 (79.7%) | 1.0: n=14 (20.3%),0.0: n=18 (85.7%) | 1.0: n=3 (14.3%),0.0: n=132 (89.2%) | 1.0: n=16 (10.8%),0.0: n=109 (79.6%) | 1.0: n=28 (20.4%)
2,comp_maternaGrave,Chi-Square,0.0000,G0vG1: 0.001 | G0vG2: 1.000 | G0vG3: 0.218 | G0vG4: 1.000 | G1vG2: 0.159 | G1vG3: 0.000 | G1vG4: 0.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 1.000,0.0: n=77 (91.7%) | 1.0: n=7 (8.3%),0.0: n=45 (65.2%) | 1.0: n=24 (34.8%),0.0: n=20 (95.2%) | 1.0: n=1 (4.8%),0.0: n=146 (98.6%) | 1.0: n=2 (1.4%),0.0: n=130 (94.9%) | 1.0: n=7 (5.1%)
3,comp_metabolica,Chi-Square,0.1056,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 0.164 | G0vG4: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G1vG4: 1.000 | G2vG3: 1.000 | G2vG4: 1.000 | G3vG4: 1.000,0.0: n=69 (82.1%) | 1.0: n=15 (17.9%),0.0: n=63 (91.3%) | 1.0: n=6 (8.7%),0.0: n=18 (85.7%) | 1.0: n=3 (14.3%),0.0: n=138 (93.2%) | 1.0: n=10 (6.8%),0.0: n=123 (89.8%) | 1.0: n=14 (10.2%)
4,ant_obstetrico,Chi-Square,0.6537,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G0vG4: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G1vG4: 1.000 | G2vG3: 0.314 | G2vG4: 1.000 | G3vG4: 1.000,-1: n=61 (72.6%) | 1: n=18 (21.4%) | 3: n=3 (3.6%) | 4: n=2 (2.4%),-1: n=52 (75.4%) | 1: n=15 (21.7%) | 3: n=2 (2.9%),-1: n=17 (81.0%) | 1: n=2 (9.5%) | 3: n=1 (4.8%) | 4: n=1 (4.8%),-1: n=106 (71.6%) | 1: n=34 (23.0%) | 3: n=8 (5.4%),-1: n=100 (73.0%) | 1: n=30 (21.9%) | 3: n=4 (2.9%) | 4: n=3 (2.2%)
5,ant_medico,Chi-Square,0.0131,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G0vG4: 0.109 | G1vG2: 1.000 | G1vG3: 1.000 | G1vG4: 0.033 | G2vG3: 0.312 | G2vG4: 0.003 | G3vG4: 1.000,-1: n=60 (71.4%) | 1: n=19 (22.6%) | 3: n=5 (6.0%),-1: n=48 (69.6%) | 1: n=19 (27.5%) | 3: n=2 (2.9%),-1: n=11 (52.4%) | 1: n=8 (38.1%) | 3: n=2 (9.5%),-1: n=117 (79.1%) | 1: n=27 (18.2%) | 3: n=3 (2.0%) | 5: n=1 (0.7%),-1: n=120 (87.6%) | 1: n=13 (9.5%) | 3: n=4 (2.9%)


# Clustering con K-Medoids

In [38]:
%pip install scikit-learn-extra gower


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
from sklearn_extra.cluster import KMedoids
import gower
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [40]:
vars_clustering_kmedoids = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM', 
    'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 
    'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica'
]

# Filtrar variables existentes y limpiar NaNs
vars_disponibles = [v for v in vars_clustering_kmedoids if v in df_maestro.columns]
df_clean = df_maestro.dropna(subset=vars_disponibles).copy()
vars_finales = [v for v in vars_disponibles if df_clean[v].nunique() > 1]

# Forzar tipos numéricos para evitar conflictos en la matriz de Gower
for v in vars_finales:
    df_clean[v] = pd.to_numeric(df_clean[v], errors='coerce').fillna(0)

# FUNCIÓN DE EVALUACIÓN 
def evaluar_kmedoids(df_input, vars_list, k):
    # Forzamos float64 para evitar el error UFuncTypeError: Cannot cast ufunc 'divide'
    X = df_input[vars_list].copy().astype(np.float64)
    cat_features = [v in variables_categoricas for v in vars_list]
    
    # Cálculo de matriz de Gower
    dist_matrix = gower.gower_matrix(X, cat_features=cat_features)
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    np.fill_diagonal(dist_matrix, 0)
    
    # Modelo K-Medoids
    km = KMedoids(n_clusters=k, metric='precomputed', method='pam', random_state=42)
    labels = km.fit_predict(dist_matrix)
    
    if len(np.unique(labels)) < 2:
        return 0.0, labels
        
    sil = silhouette_score(dist_matrix, labels, metric='precomputed')
    return sil, labels

# BÚSQUEDA DE COMBINACIONES (FORWARD SELECTION + FILTRO n>=100)
def buscar_top3_kmedoids(df_input, k_range, candidate_vars):
    all_k_results = []
    for k in k_range:
        print(f"\n>>> BUSCANDO PARA K={k} (Requisito: cada grupo n >= 100) <<<")
        tested_combinations = []
        current_vars = []
        remaining = list(candidate_vars)
        
        while remaining:
            step_scores = []
            for var in remaining:
                temp_vars = current_vars + [var]
                try:
                    sil, labels = evaluar_kmedoids(df_input, temp_vars, k)
                    counts = pd.Series(labels).value_counts().sort_index()
                    counts_list = counts.tolist()
                    
                    # Lógica de filtrado:
                    # Si ya formó K grupos, verificamos que todos tengan >= 100
                    formo_k = (len(counts_list) == k)
                    cumple_n = all(n >= 100 for n in counts_list)

                    if formo_k and cumple_n:
                        tested_combinations.append({
                            'k': k, 'silhouette': sil, 
                            'num_variables': len(temp_vars),
                            'variables': list(temp_vars),
                            'n_grupos': counts_list
                        })
                        step_scores.append((sil, var))
                    elif not formo_k:
                        # Si aún no separa en K grupos, permitimos que siga para añadir más variables
                        step_scores.append((0.0001, var))
                except: continue
            
            if not step_scores: break
            
            step_scores.sort(key=lambda x: x[0], reverse=True)
            best_sil, best_var = step_scores[0]
            if best_sil <= 0: break
                
            current_vars.append(best_var)
            remaining.remove(best_var)
            print(f" Paso {len(current_vars)}: +{best_var:<25} | Sil: {best_sil:.4f}")

        df_k = pd.DataFrame(tested_combinations)
        if not df_k.empty:
            top3 = df_k.sort_values('silhouette', ascending=False).head(3)
            all_k_results.append(top3)
        else:
            print(f"  AVISO: No se encontraron combinaciones válidas para K={k}")
    
    return pd.concat(all_k_results, ignore_index=True) if all_k_results else pd.DataFrame()

# EJECUCIÓN DE LA BÚSQUEDA
top3_kmedoids = buscar_top3_kmedoids(df_clean, [2, 3, 4], vars_finales)

# ANÁLISIS ESTADÍSTICO DE LOS RESULTADOS
if not top3_kmedoids.empty:
    print("\n" + "="*80)
    print("RESUMEN DE LAS MEJORES COMBINACIONES ENCONTRADAS")
    print("="*80)
    pd.set_option('display.max_colwidth', None)
    display(top3_kmedoids)

    for idx, row in top3_kmedoids.iterrows():
        k_val = int(row['k'])
        vars_sel = row['variables']
        
        print(f"\n\n{'#'*100}")
        print(f"ANÁLISIS DETALLADO: K={k_val} | Sil: {row['silhouette']:.4f} | Tamaños: {row['n_grupos']}")
        print(f"Variables: {vars_sel}")
        print(f"{'#'*100}")
        
        # Generar mapa de clusters para esta combinación
        _, labels = evaluar_kmedoids(df_clean, vars_sel, k_val)
        mapa = pd.DataFrame({'id': df_clean['id'], 'cluster': labels})
        
        # Unir con maestro (limpiando duplicados)
        df_base = df_maestro.loc[:, ~df_maestro.columns.duplicated()].copy()
        if 'cluster' in df_base.columns: df_base = df_base.drop(columns=['cluster'])
        df_final = pd.merge(df_base, mapa, on='id', how='inner')
        
        for nombre_bloque, vars_bloque in grupos_informe.items():
            v_existentes = [v for v in vars_bloque if v in df_final.columns]
            data_stats = []
            for v in v_existentes:
                if v not in ['id', 'ID']:
                    try:
                        res = realizar_analisis_avanzado(df_final, v, 'cluster')
                        if res: data_stats.append(res)
                    except: continue
            
            if data_stats:
                print(f"\n{nombre_bloque}")
                # Headers dinámicos para evitar errores de dimensión
                num_tot = len(data_stats[0])
                headers_fijos = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"]
                headers_dinamicos = headers_fijos + [f"G{i}" for i in range(num_tot - len(headers_fijos))]
                
                df_res = pd.DataFrame(data_stats, columns=headers_dinamicos)
                display(df_res)
else:
    print("\nERROR: No se encontró ninguna combinación que cumpla con el mínimo de 100 pacientes por grupo.")


>>> BUSCANDO PARA K=2 (Requisito: cada grupo n >= 100) <<<
 Paso 1: +trastorno_hipertensivo    | Sil: 1.0000
 Paso 2: +ratio_MoM                 | Sil: 0.9740
 Paso 3: +sflt1_MoM                 | Sil: 0.9421
 Paso 4: +apgar_5min                | Sil: 0.9105
 Paso 5: +apgar_1min                | Sil: 0.8662
 Paso 6: +apgar_10min               | Sil: 0.8260
 Paso 7: +comp_maternaGrave         | Sil: 0.7704
 Paso 8: +plgf_MoM                  | Sil: 0.7248
 Paso 9: +eg_parto                  | Sil: 0.6814
 Paso 10: +tad_1tri                  | Sil: 0.6410
 Paso 11: +aumento_peso_gest         | Sil: 0.6037
 Paso 12: +imc_ini_gest              | Sil: 0.5676
 Paso 13: +tas_1tri                  | Sil: 0.5347
 Paso 14: +peso_ini_gest             | Sil: 0.5054
 Paso 15: +peso_fin_gest             | Sil: 0.4796
 Paso 16: +peso_rn                   | Sil: 0.4563
 Paso 17: +talla                     | Sil: 0.4334
 Paso 18: +edad_materna_gest         | Sil: 0.4126
 Paso 19: +comp_metabolica     

,k,silhouette,num_variables,variables,n_grupos
0,2,1.000000,1,[trastorno_hipertensivo],"[132, 326]"
1,2,0.974011,2,"[trastorno_hipertensivo, ratio_MoM]","[132, 326]"
2,2,0.963747,2,"[trastorno_hipertensivo, sflt1_MoM]","[326, 132]"
3,3,0.547074,1,[peso_ini_gest],"[104, 150, 204]"
4,3,0.545810,1,[tas_1tri],"[197, 123, 138]"
5,3,0.536295,1,[peso_fin_gest],"[125, 112, 221]"
6,4,0.613570,2,"[trastorno_hipertensivo, talla]","[132, 119, 106, 101]"
7,4,0.581894,3,"[trastorno_hipertensivo, talla, ratio_MoM]","[110, 132, 106, 110]"
8,4,0.548163,3,"[trastorno_hipertensivo, talla, sflt1_MoM]","[109, 132, 111, 106]"




####################################################################################################
ANÁLISIS DETALLADO: K=2 | Sil: 1.0000 | Tamaños: [132, 326]
Variables: ['trastorno_hipertensivo']
####################################################################################################

GESTACIÓN Y BIOMARCADORES


,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1
0,peso_ini_gest,U Mann-Whitney,0.0040,G0vG1: 0.004,67.35 ± 21.25,63.00 ± 15.20
1,peso_fin_gest,U Mann-Whitney,0.0004,G0vG1: 0.000,82.00 ± 22.25,74.54 ± 16.00
2,aumento_peso_gest,U Mann-Whitney,0.1289,G0vG1: 0.129,12.00 ± 5.50,11.00 ± 5.50
3,talla,U Mann-Whitney,0.0067,G0vG1: 0.007,163.00 ± 8.25,164.00 ± 9.00
4,imc_ini_gest,U Mann-Whitney,0.0000,G0vG1: 0.000,25.70 ± 6.65,22.98 ± 5.52
5,edad_materna_gest,U Mann-Whitney,0.0133,G0vG1: 0.013,36.88 ± 6.86,35.31 ± 5.98
6,tas_1tri,U Mann-Whitney,0.0000,G0vG1: 0.000,116.04 ± 13.00,112.16 ± 16.00
7,tad_1tri,U Mann-Whitney,0.0000,G0vG1: 0.000,75.63 ± 9.03,72.00 ± 9.00
8,eg_eco_1tri,U Mann-Whitney,0.5746,G0vG1: 0.575,12.73 ± 0.70,12.80 ± 0.50
9,eg_parto,U Mann-Whitney,0.0000,G0vG1: 0.000,38.30 ± 2.80,39.80 ± 1.70



ECOCARDIOGRAFÍA


...

,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1,G2,G3
0,hemoglobina,Kruskal-Wallis,0.4180,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 0.861 | G1vG3: 1.000 | G2vG3: 1.000,132.00 ± 15.00,132.00 ± 12.50,127.00 ± 14.00,131.00 ± 14.00
1,hematocrito,Kruskal-Wallis,0.7137,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,0.39 ± 0.04,0.39 ± 0.03,0.39 ± 0.04,0.39 ± 0.04
2,leucocitos,Kruskal-Wallis,0.8544,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,6160.00 ± 1810.00,6310.00 ± 2150.00,6215.00 ± 1750.00,6460.00 ± 2360.00
3,plaquetas,Kruskal-Wallis,0.0858,G0vG1: 0.700 | G0vG2: 1.000 | G0vG3: 0.414 | G1vG2: 0.529 | G1vG3: 1.000 | G2vG3: 0.215,257000.00 ± 84000.00,279000.00 ± 74000.00,248500.00 ± 75000.00,278000.00 ± 68000.00
4,glucosa,Kruskal-Wallis,0.2539,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 0.338 | G2vG3: 1.000,86.00 ± 8.50,88.00 ± 10.00,86.50 ± 8.00,86.00 ± 9.00
5,sodio,Kruskal-Wallis,0.9865,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,140.00 ± 3.00,140.00 ± 2.50,140.00 ± 3.00,140.00 ± 2.00
6,potasio,Kruskal-Wallis,0.5196,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 0.819 | G2vG3: 1.000,4.22 ± 0.38,4.26 ± 0.35,4.19 ± 0.32,4.17 ± 0.27
7,urato_acidourico,Kruskal-Wallis,0.8377,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,3.98 ± 0.98,4.09 ± 1.20,4.08 ± 1.01,4.16 ± 1.23
8,hemoglobina_glicada,Kruskal-Wallis,0.0592,G0vG1: 0.100 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 0.794 | G1vG3: 0.195 | G2vG3: 1.000,5.30 ± 0.30,5.40 ± 0.30,5.30 ± 0.30,5.30 ± 0.40
9,ast,Kruskal-Wallis,0.6055,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,20.00 ± 5.50,20.00 ± 6.00,20.00 ± 6.00,20.00 ± 6.00



SCORES CATEGÓRICOS


,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1,G2,G3
0,Cat_Dieta,Chi-Square,0.6325,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,Adherencia moderada: n=66 (60.6%) | Alta adherencia: n=6 (5.5%) | Baja adherencia: n=37 (33.9%),Adherencia moderada: n=78 (59.1%) | Alta adherencia: n=6 (4.5%) | Baja adherencia: n=48 (36.4%),Adherencia moderada: n=72 (64.9%) | Alta adherencia: n=9 (8.1%) | Baja adherencia: n=30 (27.0%),Adherencia moderada: n=68 (64.2%) | Alta adherencia: n=4 (3.8%) | Baja adherencia: n=34 (32.1%)
1,Cat_Actividad,Chi-Square,0.3022,G0vG1: 0.654 | G0vG2: 0.828 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 0.703,Actividad física alta: n=101 (92.7%) | Actividad física baja: n=4 (3.7%) | Actividad física moderada: n=4 (3.7%),Actividad física alta: n=119 (92.2%) | Actividad física baja: n=1 (0.8%) | Actividad física moderada: n=9 (7.0%),Actividad física alta: n=98 (88.3%) | Actividad física baja: n=2 (1.8%) | Actividad física moderada: n=11 (9.9%),Actividad física alta: n=94 (91.3%) | Actividad física baja: n=4 (3.9%) | Actividad física moderada: n=5 (4.9%)
2,Cat_Estres,Chi-Square,0.5817,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,Nivel de estrés alto: n=42 (38.5%) | Nivel de estrés bajo: n=15 (13.8%) | Nivel de estrés moderado: n=49 (45.0%) | Nivel de estrés muy alto: n=3 (2.8%),Nivel de estrés alto: n=49 (37.4%) | Nivel de estrés bajo: n=13 (9.9%) | Nivel de estrés moderado: n=63 (48.1%) | Nivel de estrés muy alto: n=6 (4.6%),Nivel de estrés alto: n=32 (29.1%) | Nivel de estrés bajo: n=16 (14.5%) | Nivel de estrés moderado: n=58 (52.7%) | Nivel de estrés muy alto: n=4 (3.6%),Nivel de estrés alto: n=38 (35.8%) | Nivel de estrés bajo: n=8 (7.5%) | Nivel de estrés moderado: n=58 (54.7%) | Nivel de estrés muy alto: n=2 (1.9%)
3,Cat_Memoria,Chi-Square,0.4740,G0vG1: 0.651 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,Deterioro leve: n=12 (11.0%) | Disfunción moderada-grave: n=8 (7.3%) | Función normal (lapsus leves): n=65 (59.6%) | Rendimiento de memoria óptimo: n=24 (22.0%),Deterioro leve: n=25 (19.4%) | Disfunción moderada-grave: n=15 (11.6%) | Función normal (lapsus leves): n=65 (50.4%) | Rendimiento de memoria óptimo: n=24 (18.6%),Deterioro leve: n=16 (14.5%) | Disfunción moderada-grave: n=9 (8.2%) | Función normal (lapsus leves): n=65 (59.1%) | Rendimiento de memoria óptimo: n=20 (18.2%),Deterioro leve: n=11 (10.5%) | Disfunción moderada-grave: n=12 (11.4%) | Función normal (lapsus leves): n=65 (61.9%) | Rendimiento de memoria óptimo: n=17 (16.2%)
4,Cat_SCL90R,Chi-Square,0.0257,G0vG1: 0.061 | G0vG2: 1.000 | G0vG3: 0.328 | G1vG2: 0.016 | G1vG3: 1.000 | G2vG3: 0.244,EN RIESGO: n=10 (9.2%) | Normal / Sin riesgo: n=27 (24.8%) | Patología severa: n=72 (66.1%),EN RIESGO: n=17 (13.4%) | Normal / Sin riesgo: n=15 (11.8%) | Patología severa: n=95 (74.8%),EN RIESGO: n=13 (11.7%) | Normal / Sin riesgo: n=31 (27.9%) | Patología severa: n=67 (60.4%),EN RIESGO: n=17 (16.5%) | Normal / Sin riesgo: n=16 (15.5%) | Patología severa: n=70 (68.0%)



ANTECEDENTES


,Variable,Test,p-valor (Total),p-valor (dos a dos),G0,G1,G2,G3
0,trastorno_hipertensivo,Chi-Square,0.0000,G0vG1: 0.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 0.000 | G1vG3: 0.000 | G2vG3: 1.000,0.0: n=109 (100.0%),1.0: n=132 (100.0%),0.0: n=111 (100.0%),0.0: n=106 (100.0%)
1,comp_placentaria,Chi-Square,0.0362,G0vG1: 1.000 | G0vG2: 0.322 | G0vG3: 1.000 | G1vG2: 0.036 | G1vG3: 1.000 | G2vG3: 0.612,0.0: n=91 (83.5%) | 1.0: n=18 (16.5%),0.0: n=105 (79.5%) | 1.0: n=27 (20.5%),0.0: n=103 (92.8%) | 1.0: n=8 (7.2%),0.0: n=90 (84.9%) | 1.0: n=16 (15.1%)
2,comp_maternaGrave,Chi-Square,0.0000,G0vG1: 0.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 0.000 | G1vG3: 0.000 | G2vG3: 1.000,0.0: n=108 (99.1%) | 1.0: n=1 (0.9%),0.0: n=94 (71.2%) | 1.0: n=38 (28.8%),0.0: n=109 (98.2%) | 1.0: n=2 (1.8%),0.0: n=106 (100.0%)
3,comp_metabolica,Chi-Square,0.0661,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 0.301 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 0.268,0.0: n=101 (92.7%) | 1.0: n=8 (7.3%),0.0: n=118 (89.4%) | 1.0: n=14 (10.6%),0.0: n=103 (92.8%) | 1.0: n=8 (7.2%),0.0: n=88 (83.0%) | 1.0: n=18 (17.0%)
4,ant_obstetrico,Chi-Square,0.8041,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 1.000 | G2vG3: 1.000,-1: n=77 (70.6%) | 1: n=27 (24.8%) | 3: n=4 (3.7%) | 4: n=1 (0.9%),-1: n=98 (74.2%) | 1: n=27 (20.5%) | 3: n=4 (3.0%) | 4: n=3 (2.3%),-1: n=86 (77.5%) | 1: n=20 (18.0%) | 3: n=5 (4.5%),-1: n=74 (69.8%) | 1: n=25 (23.6%) | 3: n=5 (4.7%) | 4: n=2 (1.9%)
5,ant_medico,Chi-Square,0.3161,G0vG1: 1.000 | G0vG2: 1.000 | G0vG3: 1.000 | G1vG2: 1.000 | G1vG3: 0.528 | G2vG3: 1.000,-1: n=87 (79.8%) | 1: n=17 (15.6%) | 3: n=5 (4.6%),-1: n=94 (71.2%) | 1: n=31 (23.5%) | 3: n=7 (5.3%),-1: n=88 (79.3%) | 1: n=20 (18.0%) | 3: n=3 (2.7%),-1: n=86 (81.1%) | 1: n=18 (17.0%) | 3: n=1 (0.9%) | 5: n=1 (0.9%)


In [53]:
import os
import json

# ── EXPORTACIÓN DE LOS ÍNDICES 1 Y 2 DE top3_kmedoids ─────────────────────────
indices_exportar = [1, 2]   # posiciones en top3_kmedoids (0-based)

os.makedirs("exportaciones_kmedoids", exist_ok=True)

for idx in indices_exportar:

    row             = top3_kmedoids.iloc[idx]
    k_actual        = int(row['k'])
    vars_clustering = list(row['variables'])
    sil_score       = float(row['silhouette'])

    nombre_combo = f"combo_{idx}_k{k_actual}_sil{sil_score:.4f}"
    print(f"\n{'='*60}")
    print(f"EXPORTANDO POSICIÓN {idx}  |  K={k_actual}  |  Silhouette={sil_score:.4f}")
    print(f"Variables de clustering: {vars_clustering}")

    # ── 1. RE-EJECUTAR EL CLUSTERING ──────────────────────────────────────────
    sil, labels = evaluar_kmedoids(df_clean, vars_clustering, k_actual)

    # ── 2. ASIGNACIÓN DE CLUSTER A CADA PACIENTE ──────────────────────────────
    mapa = pd.DataFrame({'id': df_clean['id'].values, 'cluster': labels})

    df_base = df_maestro.loc[:, ~df_maestro.columns.duplicated()].copy()
    if 'cluster' in df_base.columns:
        df_base = df_base.drop(columns=['cluster'])
    df_pacientes = pd.merge(df_base, mapa, on='id', how='inner')

    # ── 3. METADATOS ───────────────────────────────────────────────────────────
    distribucion = df_pacientes['cluster'].value_counts().sort_index().to_dict()
    metadata = {
        "posicion_en_top3_kmedoids": idx,
        "modelo":                    "K-Medoids (PAM, distancia Gower)",
        "k":                         k_actual,
        "silhouette_score":          sil_score,
        "variables_clustering":      vars_clustering,
        "n_pacientes_total":         len(df_pacientes),
        "distribucion_clusters":     distribucion,
    }

    # ── 4. ORDENAR COLUMNAS ────────────────────────────────────────────────────
    cols_ordenadas = (
        ['id', 'cluster']
        + vars_clustering
        + [c for c in df_pacientes.columns
           if c not in ['id', 'cluster'] + vars_clustering]
    )
    cols_ordenadas = [c for c in cols_ordenadas if c in df_pacientes.columns]
    df_export = df_pacientes[cols_ordenadas].sort_values(['cluster', 'id']).reset_index(drop=True)

    # ── 5. GUARDAR ARCHIVOS ────────────────────────────────────────────────────

    # 5a. CSV con todos los pacientes
    ruta_csv = f"exportaciones_kmedoids/{nombre_combo}_pacientes.csv"
    df_export.to_csv(ruta_csv, index=False)
    print(f"  → Pacientes guardados: {ruta_csv}")

    # 5b. Excel: una hoja por cluster
    ruta_excel = f"exportaciones_kmedoids/{nombre_combo}_por_cluster.xlsx"
    with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:

        df_export.to_excel(writer, sheet_name="TODOS", index=False)

        for g in sorted(df_export['cluster'].unique()):
            df_grupo = df_export[df_export['cluster'] == g].reset_index(drop=True)
            df_grupo.to_excel(writer, sheet_name=f"Cluster_{g}", index=False)

        pd.DataFrame([metadata]).T.reset_index().rename(
            columns={'index': 'campo', 0: 'valor'}
        ).to_excel(writer, sheet_name="METADATOS", index=False)

    print(f"  → Excel guardado:     {ruta_excel}")

    # 5c. JSON con metadatos
    ruta_json = f"exportaciones_kmedoids/{nombre_combo}_metadata.json"
    with open(ruta_json, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f"  → Metadata guardada:  {ruta_json}")

    # ── 6. RESUMEN EN PANTALLA ─────────────────────────────────────────────────
    print(f"\n  Distribución de clusters:")
    for g, n in distribucion.items():
        print(f"    Cluster {g}: {n} pacientes")


EXPORTANDO POSICIÓN 1  |  K=2  |  Silhouette=0.9740
Variables de clustering: ['trastorno_hipertensivo', 'ratio_MoM']
  → Pacientes guardados: exportaciones_kmedoids/combo_1_k2_sil0.9740_pacientes.csv
  → Excel guardado:     exportaciones_kmedoids/combo_1_k2_sil0.9740_por_cluster.xlsx
  → Metadata guardada:  exportaciones_kmedoids/combo_1_k2_sil0.9740_metadata.json

  Distribución de clusters:
    Cluster 0: 132 pacientes
    Cluster 1: 326 pacientes

EXPORTANDO POSICIÓN 2  |  K=2  |  Silhouette=0.9637
Variables de clustering: ['trastorno_hipertensivo', 'sflt1_MoM']
  → Pacientes guardados: exportaciones_kmedoids/combo_2_k2_sil0.9637_pacientes.csv
  → Excel guardado:     exportaciones_kmedoids/combo_2_k2_sil0.9637_por_cluster.xlsx
  → Metadata guardada:  exportaciones_kmedoids/combo_2_k2_sil0.9637_metadata.json

  Distribución de clusters:
    Cluster 0: 326 pacientes
    Cluster 1: 132 pacientes


# Clustering con HDBSCAN

Con HDBSCAN no se hacen k clusters, sino que se analiza la estructura de densidad de los datos y determina cuántos clusters existen de forma natural.

Como los clusters no tienen por qué tener forma circular en el espacio, no tiene sentido aplicar la métrica Silhouette, por lo que aplicarmos Estabilidad Media Relativa.

In [43]:
%pip install hdbscan gower
import hdbscan


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Este código tiene min_cluster_size = 25, 30, 40. Esto obliga a que cada grupo tenga al menos 25, 30 o 40 personas. De esta manera no salen muchísimos grupos. Si se quieren sacar más grupos hay que disminuir el tamaño de los clústeres. El código está preparado para disminuir el tamaño y mostrar los p-valores 2 a 2 fácilmente (porque al aumentar la cantidad de grupos aumentan también las combinaciones 2 a 2)

In [44]:
# 1. PREPARACIÓN Y LIMPIEZA
vars_clustering = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM', 
    'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 
    'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica'
]

vars_disponibles = [v for v in vars_clustering if v in df_maestro.columns]
df_clean = df_maestro.dropna(subset=vars_disponibles).copy()
vars_finales = [v for v in vars_disponibles if df_clean[v].nunique() > 1]

# Forzar tipos numéricos para evitar conflictos en la matriz de Gower
for v in vars_finales:
    df_clean[v] = pd.to_numeric(df_clean[v], errors='coerce').fillna(0)

# 2. FUNCIÓN DE EVALUACIÓN (Métrica de Estabilidad)
def evaluar_hdbscan_estabilidad(df_input, vars_list, min_samples):
    X = df_input[vars_list].copy().astype(np.float64)
    cat_features = [v in variables_categoricas for v in vars_list]
    
    # Matriz de Gower
    dist_matrix = gower.gower_matrix(X, cat_features=cat_features).astype(np.float64)
    
    # HDBSCAN: min_cluster_size=100 asegura que los grupos detectados tengan >= 100 personas
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_samples, 
        metric='precomputed', 
        gen_min_span_tree=True
    )
    labels = clusterer.fit_predict(dist_matrix)
    
    # Conteos de grupos (Cluster -1 = Ruido)
    counts = pd.Series(labels).value_counts().sort_index().to_dict()
    
    labels_sin_ruido = labels[labels != -1]
    if len(np.unique(labels_sin_ruido)) < 2:
        return 0.0, labels, counts
    
    # Métrica de estabilidad basada en la persistencia de los clusters
    score_estabilidad = np.mean(clusterer.cluster_persistence_)
    return score_estabilidad, labels, counts

# 3. BÚSQUEDA DE LAS TOP 3 COMBINACIONES
def buscar_top3_hdbscan(df_input, samples_range, candidate_vars):
    all_results = []
    for m in samples_range:
        print(f"\n> Explorando HDBSCAN (min_cluster_size={m})...")
        tested_combinations = []
        current_vars = []
        remaining = list(candidate_vars)
        
        while remaining:
            step_scores = []
            for var in remaining:
                temp_vars = current_vars + [var]
                try:
                    if len(temp_vars) < 2:
                        score, labels, counts = 0.0, [], {}
                    else:
                        score, labels, counts = evaluar_hdbscan_estabilidad(df_input, temp_vars, m)
                    
                    # Guardamos los resultados incluyendo Estabilidad y Conteos (n)
                    tested_combinations.append({
                        'min_samples': m, 
                        'estabilidad': score, 
                        'n_grupos': counts,
                        'num_vars': len(temp_vars),
                        'variables': list(temp_vars)
                    })
                    step_scores.append((score, var))
                except: continue
            
            if not step_scores: break
            step_scores.sort(key=lambda x: x[0], reverse=True)
            best_score, best_var = step_scores[0]
            
            if best_score <= 0 and len(current_vars) > 0: break # Evitar bucles infinitos si no hay estabilidad
            
            current_vars.append(best_var)
            remaining.remove(best_var)
            if len(current_vars) > 1:
                print(f"  Paso {len(current_vars)}: +{best_var:<25} | Estabilidad: {best_score:.4f}")

        # Top 3 de este rango
        df_m = pd.DataFrame(tested_combinations).sort_values('estabilidad', ascending=False).head(3)
        all_results.append(df_m)
        
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# EJECUCIÓN (Filtro n >= 100)
top3_hdbscan = buscar_top3_hdbscan(df_clean, [100], vars_finales)

# 4. ANÁLISIS ESTADÍSTICO DETALLADO
print("\n" + "="*70)
print("RESUMEN Y ANÁLISIS ESTADÍSTICO HDBSCAN (n_min >= 100)")
print("="*70)

if not top3_hdbscan.empty:
    pd.set_option('display.max_colwidth', None)
    # Mostramos la tabla resumen inicial con la Estabilidad
    display(top3_hdbscan[['min_samples', 'estabilidad', 'n_grupos', 'num_vars', 'variables']])

    for idx, row in top3_hdbscan.iterrows():
        m_val = int(row['min_samples'])
        vars_sel = row['variables']
        est_val = row['estabilidad']
        conteos = row['n_grupos']
        
        # Formatear el n de los grupos reales (excluyendo ruido -1)
        n_str = ", ".join([f"G{k}: n={v}" for k, v in conteos.items() if k != -1])
        n_ruido = conteos.get(-1, 0)
        
        print(f"\n\n{'#'*100}")
        print(f"HDBSCAN TOP {idx+1} | Estabilidad: {est_val:.4f} | Tamaños: [{n_str}] | Ruido: n={n_ruido}")
        print(f"Variables: {vars_sel}")
        print(f"{'#'*100}")
        
        # Obtener etiquetas y mergear
        _, labels, _ = evaluar_hdbscan_estabilidad(df_clean, vars_sel, m_val)
        mapa = pd.DataFrame({'id': df_clean['id'], 'cluster': labels})
        df_base = df_maestro.loc[:, ~df_maestro.columns.duplicated()].copy()
        if 'cluster' in df_base.columns: df_base = df_base.drop(columns=['cluster'])
        df_final = pd.merge(df_base, mapa, on='id', how='inner')
        
        # Solo analizamos clusters reales (n >= 100, no ruido)
        df_stats = df_final[df_final['cluster'] != -1].copy()
        clusters_reales = sorted(df_stats['cluster'].unique())
        
        if len(clusters_reales) < 2:
            print("No hay suficientes clusters válidos para análisis estadístico.")
            continue

        for nombre_bloque, vars_bloque in grupos_informe.items():
            v_existentes = [v for v in vars_bloque if v in df_stats.columns]
            data_stats = []
            for v in v_existentes:
                if v not in ['id', 'ID']:
                    try:
                        res = realizar_analisis_avanzado(df_stats, v, 'cluster')
                        if res: data_stats.append(res)
                    except: continue
            
            if data_stats:
                print(f"\n{nombre_bloque}")
                num_col_tot = len(data_stats[0])
                headers_fijos = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"]
                headers_dinamicos = headers_fijos + [f"G{i}" for i in range(num_col_tot - len(headers_fijos))]
                
                df_res_completo = pd.DataFrame(data_stats, columns=headers_dinamicos)
                display(df_res_completo.drop(columns=["p-valor (dos a dos)"]).style.set_caption(f"Estadísticos: {nombre_bloque}"))
else:
    print("No se encontraron combinaciones que cumplan el requisito de 100 pacientes por grupo.")


> Explorando HDBSCAN (min_cluster_size=100)...
  Paso 2: +trastorno_hipertensivo    | Estabilidad: 0.4165
  Paso 3: +plgf_MoM                  | Estabilidad: 0.4299
  Paso 4: +comp_metabolica           | Estabilidad: 0.4151
  Paso 5: +sflt1_MoM                 | Estabilidad: 0.3837
  Paso 6: +ratio_MoM                 | Estabilidad: 0.3538
  Paso 7: +apgar_5min                | Estabilidad: 0.3279

RESUMEN Y ANÁLISIS ESTADÍSTICO HDBSCAN (n_min >= 100)


,min_samples,estabilidad,n_grupos,num_vars,variables
0,100,0.429904,"{0: 132, 1: 326}",3,"[peso_ini_gest, trastorno_hipertensivo, plgf_MoM]"
1,100,0.424115,"{0: 132, 1: 326}",3,"[peso_ini_gest, trastorno_hipertensivo, tad_1tri]"
2,100,0.416520,"{0: 132, 1: 326}",2,"[peso_ini_gest, trastorno_hipertensivo]"




####################################################################################################
HDBSCAN TOP 1 | Estabilidad: 0.4299 | Tamaños: [G0: n=132, G1: n=326] | Ruido: n=0
Variables: ['peso_ini_gest', 'trastorno_hipertensivo', 'plgf_MoM']
####################################################################################################

GESTACIÓN Y BIOMARCADORES


,Variable,Test,p-valor (Total),G0,G1
0,peso_ini_gest,U Mann-Whitney,0.0040,67.35 ± 21.25,63.00 ± 15.20
1,peso_fin_gest,U Mann-Whitney,0.0004,82.00 ± 22.25,74.54 ± 16.00
2,aumento_peso_gest,U Mann-Whitney,0.1289,12.00 ± 5.50,11.00 ± 5.50
3,talla,U Mann-Whitney,0.0067,163.00 ± 8.25,164.00 ± 9.00
4,imc_ini_gest,U Mann-Whitney,0.0000,25.70 ± 6.65,22.98 ± 5.52
5,edad_materna_gest,U Mann-Whitney,0.0133,36.88 ± 6.86,35.31 ± 5.98
6,tas_1tri,U Mann-Whitney,0.0000,116.04 ± 13.00,112.16 ± 16.00
7,tad_1tri,U Mann-Whitney,0.0000,75.63 ± 9.03,72.00 ± 9.00
8,eg_eco_1tri,U Mann-Whitney,0.5746,12.73 ± 0.70,12.80 ± 0.50
9,eg_parto,U Mann-Whitney,0.0000,38.30 ± 2.80,39.80 ± 1.70



ECOCARDIOGRAFÍA


...

,Variable,Test,p-valor (Total),G0,G1
0,hemoglobina,U Mann-Whitney,0.3803,132.00 ± 12.50,130.00 ± 14.00
1,hematocrito,U Mann-Whitney,0.5287,0.39 ± 0.03,0.39 ± 0.04
2,leucocitos,U Mann-Whitney,0.8963,6310.00 ± 2150.00,6300.00 ± 1920.00
3,plaquetas,U Mann-Whitney,0.2276,279000.00 ± 74000.00,261000.00 ± 81750.00
4,glucosa,U Mann-Whitney,0.0988,88.00 ± 10.00,86.00 ± 9.00
5,sodio,U Mann-Whitney,0.9685,140.00 ± 2.50,140.00 ± 3.00
6,potasio,U Mann-Whitney,0.2004,4.26 ± 0.35,4.18 ± 0.31
7,urato_acidourico,U Mann-Whitney,0.9312,4.09 ± 1.20,4.03 ± 1.11
8,hemoglobina_glicada,U Mann-Whitney,0.0113,5.40 ± 0.30,5.30 ± 0.30
9,ast,U Mann-Whitney,0.4128,20.00 ± 6.00,20.00 ± 6.00



SCORES CATEGÓRICOS


,Variable,Test,p-valor (Total),G0,G1
0,Cat_Dieta,Chi-Square,0.5030,Adherencia moderada: n=78 (59.1%) | Alta adherencia: n=6 (4.5%) | Baja adherencia: n=48 (36.4%),Adherencia moderada: n=206 (63.2%) | Alta adherencia: n=19 (5.8%) | Baja adherencia: n=101 (31.0%)
1,Cat_Actividad,Chi-Square,0.3409,Actividad física alta: n=119 (92.2%) | Actividad física baja: n=1 (0.8%) | Actividad física moderada: n=9 (7.0%),Actividad física alta: n=293 (90.7%) | Actividad física baja: n=10 (3.1%) | Actividad física moderada: n=20 (6.2%)
2,Cat_Estres,Chi-Square,0.6488,Nivel de estrés alto: n=49 (37.4%) | Nivel de estrés bajo: n=13 (9.9%) | Nivel de estrés moderado: n=63 (48.1%) | Nivel de estrés muy alto: n=6 (4.6%),Nivel de estrés alto: n=112 (34.5%) | Nivel de estrés bajo: n=39 (12.0%) | Nivel de estrés moderado: n=165 (50.8%) | Nivel de estrés muy alto: n=9 (2.8%)
3,Cat_Memoria,Chi-Square,0.1246,Deterioro leve: n=25 (19.4%) | Disfunción moderada-grave: n=15 (11.6%) | Función normal (lapsus leves): n=65 (50.4%) | Rendimiento de memoria óptimo: n=24 (18.6%),Deterioro leve: n=39 (12.0%) | Disfunción moderada-grave: n=29 (9.0%) | Función normal (lapsus leves): n=195 (60.2%) | Rendimiento de memoria óptimo: n=61 (18.8%)
4,Cat_SCL90R,Chi-Square,0.0284,EN RIESGO: n=17 (13.4%) | Normal / Sin riesgo: n=15 (11.8%) | Patología severa: n=95 (74.8%),EN RIESGO: n=40 (12.4%) | Normal / Sin riesgo: n=74 (22.9%) | Patología severa: n=209 (64.7%)



ANTECEDENTES


,Variable,Test,p-valor (Total),G0,G1
0,trastorno_hipertensivo,Chi-Square,0.0000,1.0: n=132 (100.0%),0.0: n=326 (100.0%)
1,comp_placentaria,Chi-Square,0.0565,0.0: n=105 (79.5%) | 1.0: n=27 (20.5%),0.0: n=284 (87.1%) | 1.0: n=42 (12.9%)
2,comp_maternaGrave,Chi-Square,0.0000,0.0: n=94 (71.2%) | 1.0: n=38 (28.8%),0.0: n=323 (99.1%) | 1.0: n=3 (0.9%)
3,comp_metabolica,Chi-Square,1.0000,0.0: n=118 (89.4%) | 1.0: n=14 (10.6%),0.0: n=292 (89.6%) | 1.0: n=34 (10.4%)
4,ant_obstetrico,Chi-Square,0.6062,-1: n=98 (74.2%) | 1: n=27 (20.5%) | 3: n=4 (3.0%) | 4: n=3 (2.3%),-1: n=237 (72.7%) | 1: n=72 (22.1%) | 3: n=14 (4.3%) | 4: n=3 (0.9%)
5,ant_medico,Chi-Square,0.1523,-1: n=94 (71.2%) | 1: n=31 (23.5%) | 3: n=7 (5.3%),-1: n=261 (80.1%) | 1: n=55 (16.9%) | 3: n=9 (2.8%) | 5: n=1 (0.3%)
